In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

### Bistability

In [ ]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_):
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if aln.params[init_vars[iv]].ndim == 2:
                    aln.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    aln.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
max_c_c = 500.
min_c_c = - 500.
max_c_r = 0.18
min_c_r = 0.

def setmaxmincontrol(cntrl_vars, max_c_c, min_c_c, max_c_r, min_c_r):
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

In [ ]:
tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

In [ ]:
##### LOAD BOUNDARIES
with open('bistability_exc_inh.pickle','rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
plt.scatter(exc, inh)

In [ ]:
#params_bistability_exc.pop()
#params_bistability_inh.pop()

print(len(params_bistability_exc))
print(params_bistability_exc[-3:])
print(params_bistability_inh[-3:])

aln.params['duration'] = 4000.

control0 = aln.getZeroControl()
target = aln.getZeroTarget()
control0 = step_control(maxI_ = 3.)

aln.params.ext_exc_current = 0.5 * 5.
aln.params.ext_inh_current = 0.4 * 5.


plotFunc.plot_traces(aln, control0)
    
steady_rates = np.zeros((2, 2))
steady_rates[0,0] = aln.rates_exc[0,-1] # high state exc
steady_rates[0,1] = aln.rates_inh[0,-1] # high state inh

high_state_vars = np.zeros(( len(state_vars) ))
for i in range(len(state_vars)):
    if aln.state[state_vars[i]].size == 1:
        high_state_vars[i] = aln.state[state_vars[i]][0] 
    else:
        high_state_vars[i] = aln.state[state_vars[i]][0,-1]
        
control0 = step_control(maxI_ = -3.)
plotFunc.plot_traces(aln, control0, path_=path, filename_="bistability")

low_state_vars = np.zeros(( len(state_vars) ))
for i in range(len(state_vars)):
    if aln.state[state_vars[i]].size == 1:
        low_state_vars[i] = aln.state[state_vars[i]][0]
    else:
        low_state_vars[i] = aln.state[state_vars[i]][0,-1]
        
steady_rates[1,0] = aln.rates_exc[0,-1] # low state exc
steady_rates[1,1] = aln.rates_inh[0,-1] # low state inh

print(steady_rates)

plt.plot(np.arange(0,100, 0.1), aln.rates_exc[0,-1000:])      
plt.plot(np.arange(0,100, 0.1), aln.rates_inh[0,-1000:])


In [ ]:
c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

trans_time_array = np.zeros(( len(c_var) ))
trans_time_array[:] = 0.8

In [ ]:
dur = 100
dur_pre = 10
dur_post = 10

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

sheet = "10010_"
max_it = 2000

# done: 0,5,10,15,20
# inh only: 0,5,10,15,20

#target = [None] * len(params_bistability_exc)

In [ ]:
cntrl_vars = c_var[0]
prec_vars = p_var[0]

trans_time = trans_time_array[0]
max_cntrl, min_cntrl = setmaxmincontrol(cntrl_vars)

i_range = range(0, len(params_bistability_exc),29)

factor_iteration = 4

prev_i = 0

for i in i_range:
    print("------- ", i, params_bistability_exc[i], params_bistability_inh[i])
    aln.params.ext_exc_current = params_bistability_exc[i]
    aln.params.ext_inh_current = params_bistability_inh[i]
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = step_control(maxI_ = -3.)

    plotFunc.plot_traces(aln, control0)
    
    steady_rates = np.zeros((2, 2))
    steady_rates[0,0] = aln.rates_exc[0,-1] # high state exc
    steady_rates[0,1] = aln.rates_inh[0,-1] # high state inh

    control0 = step_control(maxI_ = 3.)
    plotFunc.plot_traces(aln, control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    # set low state rates as target
    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = steady_rates[0,0]
    target[i][:,1,:] = steady_rates[0,1]
    
    case = sheet + str(0) + "_init"
    
    cost.setParams(1.0, 0.0, 1.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i])
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    weights_init[i] = cost.getParams()

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]
    
##### initial guess
    weight_s = 10
    print("weight sparsity = ", weight_s)
    cost.setParams(1.0, 0.0, weight_s)

    setinit(initVars[i])
    if prev_i != 0:
        control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_s = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight sparsity = ", weight_s)
    cost.setParams(1.0, 0.0, weight_s)

    setinit(initVars[i])
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

In [ ]:
#plot initial guesses

for i in i_range[:4]:
    
    aln.params.ext_exc_current = params_bistability_exc[i]
    aln.params.ext_inh_current = params_bistability_inh[i]

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], path, filename_ = case, transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)

In [ ]:
for k in range(10):

    factor_iteration = 6

    for i in i_range:
        
        #if i not in [174]:
        #    continue
        

        print("------- ", i, params_bistability_exc[i], params_bistability_inh[i])
        aln.params.ext_exc_current = params_bistability_exc[i]
        aln.params.ext_inh_current = params_bistability_inh[i]

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_s = weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j] - 1
        print("weight sparsity = ", weight_s)
        cost.setParams(1.0, 0.0, weight_s)

        setinit(initVars[i])
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 250 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
with open('control_init_10010.pickle','wb') as f:
    pickle.dump([bestControl_init, costnode_init, weights_init], f)
    
with open('control_init_10010.pickle','rb') as f:
    load_array = pickle.load(f)
    
print(len(load_array))

#bestControl_init = load_array[0]
#costnode_init = load_array[1]
#weights_init = load_array[2]

In [ ]:
case = sheet + str(0) + "_sparse_control"
cntrl_vars = c_var[2]
prec_vars = p_var[0]

trans_time = trans_time_array[2]
max_cntrl, min_cntrl = setmaxmincontrol(cntrl_vars)

factor_iteration = 5.
    
for i in i_range:
    print("------- ", i, params_bistability_exc[i], params_bistability_inh[i])
    aln.params.ext_exc_current = params_bistability_exc[i]
    aln.params.ext_inh_current = params_bistability_inh[i]

# exc and inh control current 

    setinit(initVars[i])
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 25 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_s = 100
    print("weight sparsity = ", weight_s)
    cost.setParams(1.0, 0.0, weight_s)

    setinit(initVars[i])
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 25 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_s = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight sparsity = ", weight_s)
    cost.setParams(1.0, 0.0, weight_s)

    setinit(initVars[i])
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 250 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
for i in i_range:
    
    aln.params.ext_exc_current = params_bistability_exc[i]
    aln.params.ext_inh_current = params_bistability_inh[i]

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], path, filename_ = case, transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)])

In [ ]:
factor_iteration = 4
    
for i in i_range:

    
    print("------- ", i, params_bistability_exc[i], params_bistability_inh[i])
    aln.params.ext_exc_current = params_bistability_exc[i]
    aln.params.ext_inh_current = params_bistability_inh[i]
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_s = weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight sparsity = ", weight_s)
    cost.setParams(1.0, 0.0, weight_s)

    setinit(initVars[i])
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 250 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
with open('control_0_10010.pickle','wb') as f:
    pickle.dump([bestControl_0, costnode_0, weights_0], f)
    
with open('control_0_10010.pickle','rb') as f:
    load_array = pickle.load(f)
    
print(len(load_array))

#bestControl_0 = load_array[0]
#costnode_0 = load_array[1]
#weights_0 = load_array[2]

In [ ]:
not_checked = []
exc_ = []
inh_ = []
e_i = []

for i in range(len(params_bistability_exc)):
    if type(bestControl_0[i]) is type(None):
        print(i, " not checked yet")
        not_checked.append(i)
        continue
    elif np.amax(np.abs(bestControl_0[i][0,0,:])) < 1e-8 and np.amax(np.abs(bestControl_0[i][0,1,:])) > 1e-8:
        inh_.append(i)
        print(i, " only inhibitory current")
    elif np.amax(np.abs(bestControl_0[i][0,1,:])) < 1e-8 and np.amax(np.abs(bestControl_0[i][0,0,:])) > 1e-8:
        exc_.append(i)
        print(i, " only excitatory current")
    elif np.amax(np.abs(bestControl_0[i][0,0,:])) < 1e-8 and np.amax(np.abs(bestControl_0[i][0,1,:])) < 1e-8:
        print(i, "no control input")
    elif np.amax(np.abs(bestControl_0[i][0,0,:])) > 1e-8 and np.amax(np.abs(bestControl_0[i][0,1,:])) > 1e-8:
        print(i, " control input in both nodes")
        e_i.append(i)
    else:
        print(i, " no category")
        
print(exc_, inh_, e_i)

In [ ]:
boundary_x = [3.65, 3.5999999999999996, 3.55, 3.5, 3.4499999999999997, 3.4000000000000004, 3.35, 3.3000000000000003, 3.25, 3.2, 3.15, 3.1, 3.05, 3.0, 2.9499999999999997, 2.9, 2.8499999999999996, 2.8000000000000003, 2.75, 2.7, 2.6500000000000004, 2.6, 2.55, 2.5, 2.45, 2.4, 2.3499999999999996, 2.3000000000000003, 2.25, 2.2, 2.15, 2.1, 2.05, 2.0, 2.0, 2.0, 2.0, 2.0, 2.05, 2.05, 2.05, 2.05, 2.05, 2.05, 2.05, 2.05, 2.05, 2.05, 2.05, 2.1, 2.1, 2.1, 2.1, 2.1, 2.1, 2.1, 2.15, 2.15, 2.15, 2.15, 2.15, 2.15, 2.15, 2.15, 2.15, 2.2, 2.2, 2.2, 2.2]
boundary_y = [7.4, 5.45, 4.5, 3.95, 3.55, 3.3000000000000003, 3.1, 2.9499999999999997, 2.8000000000000003, 2.7, 2.6, 2.5, 2.45, 2.4, 2.3499999999999996, 2.25, 2.2, 2.15, 2.1, 2.1, 2.05, 2.05, 2.0, 2.0, 1.9500000000000002, 1.9500000000000002, 1.9, 1.85, 1.85, 1.85, 1.7999999999999998, 1.75, 1.75, 1.7000000000000002, 1.75, 1.7999999999999998, 1.85, 1.9, 1.9500000000000002, 2.0, 2.05, 2.1, 2.15, 2.2, 2.25, 2.3000000000000003, 2.3499999999999996, 2.4, 2.45, 2.5, 2.55, 2.6, 2.75, 3.0, 3.25, 3.4000000000000004, 3.4499999999999997, 3.75, 4.0, 4.25, 4.5, 4.75, 5.0, 5.25, 5.3500000000000005, 5.4, 5.8999999999999995, 6.4, 6.8999999999999995]

In [ ]:
yshift = [0,0.0,0.,0,0,0.0,0.0,0.]
plt.figure(figsize=(8,16))  
plt.title("Low => high, inhibitory current as initial guess, unlimited control input")

for j in range(len(boundary_x)):
    plt.scatter(boundary_x[j], boundary_y[j], s=6, c="grey")

if len(exc_) > 0:
    plt.scatter(params_bistability_exc[exc_[0]], params_bistability_inh[exc_[0]], s=30, c="red",
                label="excitatory control current")
    lenx = np.amax(bestControl_0[exc_[0]][0,0,:])
    if np.abs(np.amin(bestControl_0[exc_[0]][0,0,:])) > np.abs(lenx):
        lenx = np.amin(bestControl_0[exc_[0]][0,0,:])
    leny = np.amax(bestControl_0[exc_[0]][0,1,:])
    if np.abs(np.amin(bestControl_0[exc_[0]][0,1,:])) > np.abs(leny):
        leny = np.amin(bestControl_0[exc_[0]][0,1,:])
    plt.arrow(params_bistability_exc[exc_[0]], params_bistability_inh[exc_[0]], lenx, leny,
              head_width=0.05, length_includes_head= True, color="black")
if len(exc_) > 1:
    for i in exc_[1:]:
        plt.scatter(params_bistability_exc[i], params_bistability_inh[i], s=30, c="red")
        lenx = np.amax(bestControl_0[i][0,0,:])
        if np.abs(np.amin(bestControl_0[i][0,0,:])) > np.abs(lenx):
            lenx = np.amin(bestControl_0[i][0,0,:])
        leny = np.amax(bestControl_0[i][0,1,:])
    if np.abs(np.amin(bestControl_0[i][0,1,:])) > np.abs(leny):
        leny = np.amin(bestControl_0[i][0,1,:])
        plt.arrow(params_bistability_exc[i], params_bistability_inh[i] + yshift[i], lenx, leny,
                  head_width=0.05, length_includes_head= True, color="black")

if len(inh_) > 0:
    plt.scatter(params_bistability_exc[inh_[0]], params_bistability_inh[inh_[0]], s=30, c="blue",
                label="inhibitory control current")
    lenx = np.amax(bestControl_0[inh_[0]][0,0,:])
    if np.abs(np.amin(bestControl_0[inh_[0]][0,0,:])) > np.abs(lenx):
        lenx = np.amin(bestControl_0[inh_[0]][0,0,:])
    leny = np.amax(bestControl_0[inh_[0]][0,1,:])
    if np.abs(np.amin(bestControl_0[inh_[0]][0,1,:])) > np.abs(leny):
        leny = np.amin(bestControl_0[inh_[0]][0,1,:])
    plt.arrow(params_bistability_exc[inh_[0]], params_bistability_inh[inh_[0]], lenx, leny,
              head_width=0.05, length_includes_head= True, color="black")
if len(inh_) > 1:
    for i in inh_[1:]:
        plt.scatter(params_bistability_exc[i], params_bistability_inh[i], s=30, c="blue")
        lenx = np.amax(bestControl_0[i][0,0,:])
        if np.abs(np.amin(bestControl_0[i][0,0,:])) > np.abs(lenx):
            lenx = np.amin(bestControl_0[i][0,0,:])
        leny = np.amax(bestControl_0[i][0,1,:])
        if np.abs(np.amin(bestControl_0[i][0,1,:])) > np.abs(leny):
            leny = np.amin(bestControl_0[i][0,1,:])
        print(i,lenx,leny)
        plt.arrow(params_bistability_exc[i], params_bistability_inh[i] + yshift[i], lenx, leny,
                head_width=0.05, length_includes_head= True, color="black")
        
if len(e_i) > 0:
    plt.scatter(params_bistability_exc[e_i[0]], params_bistability_inh[e_i[0]], s=30, c="orange",
                label="control current in both nodes")
    lenx = np.amax(bestControl_0[e_i[0]][0,0,:])
    if np.abs(np.amin(bestControl_0[e_i[0]][0,0,:])) > np.abs(lenx):
        lenx = np.amin(bestControl_0[e_i[0]][0,0,:])
    leny = np.amax(bestControl_0[e_i[0]][0,1,:])
    if np.abs(np.amin(bestControl_0[e_i[0]][0,1,:])) > np.abs(leny):
        leny = np.amin(bestControl_0[e_i[0]][0,1,:])
    print(lenx,leny)
    plt.arrow(params_bistability_exc[e_i[0]], params_bistability_inh[e_i[0]], lenx, 0,
              head_width=0.05, length_includes_head= True, color="black")
    plt.arrow(params_bistability_exc[e_i[0]], params_bistability_inh[e_i[0]], 0, leny,
              head_width=0.05, length_includes_head= True, color="black")
if len(e_i) > 1:
    for i in e_i[1:]:
        plt.scatter(params_bistability_exc[i], params_bistability_inh[i], s=30, c="orange")
        lenx = np.amax(bestControl_0[i][0,0,:])
        if np.abs(np.amin(bestControl_0[i][0,0,:])) > np.abs(lenx):
            lenx = np.amin(bestControl_0[i][0,0,:])
        leny = np.amax(bestControl_0[i][0,1,:])
        if np.abs(np.amin(bestControl_0[i][0,1,:])) > np.abs(leny):
            leny = np.amin(bestControl_0[i][0,1,:])
        plt.arrow(params_bistability_exc[i], params_bistability_inh[i] + yshift[i], lenx, 0,
                head_width=0.05, length_includes_head= True, color="black")
        plt.arrow(params_bistability_exc[i], params_bistability_inh[i] + yshift[i], 0, leny,
                head_width=0.05, length_includes_head= True, color="black")

plt.xlabel("external excitatory current [nA]")
plt.ylabel("external inhibitory current [nA]")
plt.xlim([1.5,4.5])
plt.ylim([1.5,7.5])
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.13))
plt.savefig("b_2")
plt.show()

In [ ]:
for c_ in np.arange(0.43, 0.61, 0.01):
    c_e = c_ * 5.
    c_i = 0.51 * 5.

    params_bistability_exc.append(c_e)
    params_bistability_inh.append(c_i)

    bestControl_init.append(None)
    bestState_init.append(None)
    cost_init.append(None)
    runtime_init.append(None)
    grad_init.append(None)
    phi_init.append(None)
    costnode_init.append(None)
    weights_init.append(None)

    initVars.append(None)
    target.append(None)
    cost_uncontrolled.append(None)

    bestControl_0.append(None)
    bestState_0.append(None)
    cost_0.append(None)
    runtime_0.append(None)
    grad_0.append(None)
    phi_0.append(None)
    costnode_0.append(None)
    weights_0.append(None)

In [ ]:
print(params_bistability_inh[-2]/5.)

In [ ]:
bestControl_init.pop()
bestState_init.pop()
cost_init.pop()
runtime_init.pop()
grad_init.pop()
phi_init.pop()
costnode_init.pop()
weights_init.pop()

initVars.pop()
target.pop()
cost_uncontrolled.pop()

bestControl_0.pop()
bestState_0.pop()
cost_0.pop()
runtime_0.pop()
grad_0.pop()
phi_0.pop()
costnode_0.pop()
weights_0.pop()

In [ ]:
# basin of attraction for initial parameters as in high state (except for rates)

i_range = range(len(params_bistability_exc))

factor_iteration = 2

for i in i_range:
    print("------- ", i, params_bistability_exc[i], params_bistability_inh[i])
    aln.params.ext_exc_current = params_bistability_exc[i]
    aln.params.ext_inh_current = params_bistability_inh[i]
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = step_control(maxI_ = -3.)

    plotFunc.plot_traces(aln, control0)
    
    steady_rates = np.zeros((2, 2))
    steady_rates[0,0] = aln.rates_exc[0,-1] # high state exc
    steady_rates[0,1] = aln.rates_inh[0,-1] # high state inh

    control0 = step_control(maxI_ = 3.)
    plotFunc.plot_traces(aln, control0, path_=path, filename_="bistability")
    
    steady_rates[0,0] = aln.rates_exc[0,-1] # high state exc
    steady_rates[0,1] = aln.rates_inh[0,-1] # high state inh

    high_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            high_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            high_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = high_state_vars
    
